In [1]:
import pandas as pd

df = pd.read_csv('/content/olist_cleaned_dataset.csv')
display(df.head())

,customer_unique_id,num_orders,total_spent,avg_order_value,avg_item_price_per_order,avg_items_per_order,avg_review,avg_installments,city,state,...,cat_pca_1,cat_pca_2,cat_pca_3,cat_pca_4,cat_pca_5,cat_pca_6,cat_pca_7,cat_pca_8,cat_pca_9,cat_pca_10
0,0000366f3b9a7992bf8c76cfdf3221e2,-0.159829,-0.015717,0.003713,0.021035,-0.172165,0.663285,1.898826,cajamar,SP,...,-0.637624,-0.327896,-0.102259,0.059743,-0.020116,0.189493,-0.198020,0.051733,0.008437,0.073418
1,0000b849f77a49e4a4ce2b2a4ca5be3f,-0.159829,-0.574829,-0.570550,-0.561497,-0.172165,-0.120408,-0.707834,osasco,SP,...,-0.209777,0.588827,0.471880,-0.118288,-0.149629,0.046770,-0.092128,0.047734,-0.035847,0.026227
2,0000f46a3911fa3c0805444483337064,-0.159829,-0.322473,-0.311356,-0.298570,-0.172165,-0.904101,1.898826,sao jose,SC,...,-0.018761,-0.047356,-0.095521,0.369686,0.141457,-0.068647,0.166585,0.079676,0.160974,-0.034467
3,0000f6ccb0745a6a4b88665a16c9f078,-0.159829,-0.539117,-0.533870,-0.524289,-0.172165,-0.120408,0.409306,belem,PA,...,0.146240,0.166802,-0.219602,0.525551,-0.100747,-0.413076,-0.126003,0.418943,-0.255063,0.020480
4,0004aac84e0df4da2b147fca70cf8255,-0.159829,0.236639,0.262908,0.283963,-0.172165,0.663285,1.154066,sorocaba,SP,...,0.146240,0.166802,-0.219602,0.525551,-0.100747,-0.413076,-0.126003,0.418943,-0.255063,0.020480


In [ ]:
"""
Olist Customer Segmentation - Comprehensive Clustering Analysis
Complete comparative study of clustering techniques
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Create output directories
import os
for directory in ['plots', 'results']:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created '{directory}' directory")

print("=" * 100)
print("OLIST CUSTOMER SEGMENTATION - COMPREHENSIVE CLUSTERING ANALYSIS")
print("=" * 100)

# ============================================================================
# 1. LOAD AND EXPLORE THE CLEANED DATASET
# ============================================================================
print("\n" + "=" * 100)
print("1. LOADING CLEANED CUSTOMER DATASET")
print("=" * 100)

# Load the data
data_path = "/content/olist_cleaned_dataset.csv"
df = pd.read_csv(data_path)

print(f"\n✓ Dataset loaded successfully!")
print(f"  Shape: {df.shape}")
print(f"  Customers: {df.shape[0]:,}")
print(f"  Features: {df.shape[1]}")

# Display basic info
print(f"\n📊 Dataset Overview:")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📋 Column List:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n🔍 First few rows:")
print(df.head())

print(f"\n📊 Data Types:")
print(df.dtypes.value_counts())

print(f"\n❓ Missing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    missing_df = pd.DataFrame({
        'Column': missing[missing > 0].index,
        'Missing': missing[missing > 0].values,
        'Percentage': 100 * missing[missing > 0] / len(df)
    })
    print(missing_df.to_string(index=False))
else:
    print("  ✓ No missing values!")

# ============================================================================
# 2. DESCRIPTIVE STATISTICS & EDA
# ============================================================================
print("\n" + "=" * 100)
print("2. DESCRIPTIVE STATISTICS")
print("=" * 100)

# Identify feature types
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 Feature Types:")
print(f"  Numeric features: {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")

# Remove customer_unique_id from analysis features
if 'customer_unique_id' in numeric_features:
    numeric_features.remove('customer_unique_id')

print(f"\n📈 Numeric Features Summary:")
print(df[numeric_features].describe().T)

# Categorical features analysis
if categorical_features:
    print(f"\n📊 Categorical Features:")
    for col in categorical_features:
        if col != 'customer_unique_id':
            print(f"\n  {col}:")
            value_counts = df[col].value_counts().head(10)
            for val, count in value_counts.items():
                print(f"    {val}: {count} ({100*count/len(df):.2f}%)")

# Key business metrics
print(f"\n💼 KEY BUSINESS METRICS:")
print(f"  Average orders per customer: {df['num_orders'].mean():.2f}")
print(f"  Average total spent: R$ {df['total_spent'].mean():.2f}")
print(f"  Average order value: R$ {df['avg_order_value'].mean():.2f}")
print(f"  Average review score: {df['avg_review'].mean():.2f}/5")
print(f"  Average installments: {df['avg_installments'].mean():.2f}")

# ============================================================================
# 3. VISUALIZATIONS - EXPLORATORY DATA ANALYSIS
# ============================================================================
print("\n" + "=" * 100)
print("3. CREATING EDA VISUALIZATIONS")
print("=" * 100)

# VISUALIZATION 1: Distribution of Key Metrics
print("\n📊 Creating: Distribution of Key Metrics...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution of Key Customer Metrics', fontsize=18, fontweight='bold', y=1.00)

key_metrics = ['num_orders', 'total_spent', 'avg_order_value',
               'avg_review', 'avg_installments', 'avg_items_per_order']
colors = ['steelblue', 'coral', 'lightgreen', 'gold', 'mediumpurple', 'lightcoral']

for idx, (metric, color) in enumerate(zip(key_metrics, colors)):
    ax = axes[idx // 3, idx % 3]
    data = df[metric].dropna()

    # Remove extreme outliers for better visualization
    q99 = data.quantile(0.99)
    data_filtered = data[data <= q99]

    ax.hist(data_filtered, bins=50, color=color, edgecolor='black', linewidth=0.8, alpha=0.7)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='blue', linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
    ax.set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Value', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/01_key_metrics_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/01_key_metrics_distribution.png")

# VISUALIZATION 2: Geographic Distribution
print("\n📊 Creating: Geographic Distribution...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 states
state_dist = df['state'].value_counts().head(15)
colors_geo = sns.color_palette("viridis", len(state_dist))
bars = axes[0].barh(range(len(state_dist)), state_dist.values, color=colors_geo, edgecolor='black', linewidth=1)
axes[0].set_title('Top 15 States by Customer Count', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Number of Customers', fontsize=12)
axes[0].set_ylabel('State', fontsize=12)
axes[0].set_yticks(range(len(state_dist)))
axes[0].set_yticklabels(state_dist.index)
axes[0].invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, state_dist.values)):
    width = bar.get_width()
    axes[0].text(width, bar.get_y() + bar.get_height()/2.,
                 f' {val:,}', ha='left', va='center', fontsize=10, fontweight='bold')

# Top 15 cities
city_dist = df['city'].value_counts().head(15)
bars2 = axes[1].barh(range(len(city_dist)), city_dist.values, color=colors_geo, edgecolor='black', linewidth=1)
axes[1].set_title('Top 15 Cities by Customer Count', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Number of Customers', fontsize=12)
axes[1].set_ylabel('City', fontsize=12)
axes[1].set_yticks(range(len(city_dist)))
axes[1].set_yticklabels(city_dist.index)
axes[1].invert_yaxis()

for i, (bar, val) in enumerate(zip(bars2, city_dist.values)):
    width = bar.get_width()
    axes[1].text(width, bar.get_y() + bar.get_height()/2.,
                 f' {val:,}', ha='left', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/02_geographic_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/02_geographic_distribution.png")

# VISUALIZATION 3: Correlation Heatmap
print("\n📊 Creating: Correlation Heatmap...")
# Select non-PCA numeric features for correlation
non_pca_features = [col for col in numeric_features if not col.startswith('cat_pca_')]
corr_matrix = df[non_pca_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            annot_kws={'fontsize': 8, 'fontweight': 'bold'})
plt.title('Correlation Heatmap - Customer Features', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('plots/03_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/03_correlation_heatmap.png")

# VISUALIZATION 4: Box Plots for Key Metrics
print("\n📊 Creating: Box Plots...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Box Plots - Key Customer Metrics', fontsize=18, fontweight='bold', y=1.00)

for idx, (metric, color) in enumerate(zip(key_metrics, colors)):
    ax = axes[idx // 3, idx % 3]
    data = df[metric].dropna()

    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5,
                     boxprops=dict(facecolor=color, alpha=0.7),
                     medianprops=dict(color='red', linewidth=2),
                     whiskerprops=dict(linewidth=1.5),
                     capprops=dict(linewidth=1.5))

    ax.set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    # Add statistics text
    stats_text = f"Mean: {data.mean():.2f}\nMedian: {data.median():.2f}\nStd: {data.std():.2f}"
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('plots/04_boxplots_key_metrics.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/04_boxplots_key_metrics.png")

# ============================================================================
# 4. PREPARE DATA FOR CLUSTERING
# ============================================================================
print("\n" + "=" * 100)
print("4. PREPARING DATA FOR CLUSTERING")
print("=" * 100)

# Select features for clustering (exclude identifiers and categorical)
clustering_features = [col for col in numeric_features
                       if col not in ['customer_unique_id']]

print(f"\n📊 Features selected for clustering: {len(clustering_features)}")
for i, feat in enumerate(clustering_features, 1):
    print(f"  {i:2d}. {feat}")

# Prepare feature matrix
X = df[clustering_features].values
print(f"\n✓ Feature matrix shape: {X.shape}")

# Handle missing values if any
if np.isnan(X).any():
    print(f"  ⚠️  Found {np.isnan(X).sum()} missing values - imputing with median...")
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    print("  ✓ Missing values imputed")

# Standardize features
print(f"\n🔄 Standardizing features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("  ✓ Features standardized (mean=0, std=1)")

# Save customer IDs for later reference
customer_ids = df['customer_unique_id'].values

# ============================================================================
# 5. OPTIMAL NUMBER OF CLUSTERS - ELBOW METHOD & SILHOUETTE
# ============================================================================
print("\n" + "=" * 100)
print("5. DETERMINING OPTIMAL NUMBER OF CLUSTERS")
print("=" * 100)

print("\n🔍 Testing K-Means for k=2 to k=10...")
k_range = range(2, 11)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled, labels))
    calinski_harabasz_scores.append(calinski_harabasz_score(X_scaled, labels))

    print(f"  k={k}: Silhouette={silhouette_scores[-1]:.4f}, "
          f"Davies-Bouldin={davies_bouldin_scores[-1]:.4f}, "
          f"Calinski-Harabasz={calinski_harabasz_scores[-1]:.2f}")

# VISUALIZATION 5: Elbow Method and Metrics
print("\n📊 Creating: Optimal Clusters Visualization...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Optimal Number of Clusters Analysis', fontsize=18, fontweight='bold')

# Elbow plot
axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 0].set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticks(k_range)

# Silhouette score
axes[0, 1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[0, 1].set_title('Silhouette Score (Higher is Better)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticks(k_range)
best_k_silhouette = k_range[np.argmax(silhouette_scores)]
axes[0, 1].axvline(best_k_silhouette, color='red', linestyle='--', linewidth=2,
                   label=f'Best k={best_k_silhouette}')
axes[0, 1].legend()

# Davies-Bouldin score
axes[1, 0].plot(k_range, davies_bouldin_scores, 'ro-', linewidth=2, markersize=8)
axes[1, 0].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 0].set_ylabel('Davies-Bouldin Score', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticks(k_range)
best_k_db = k_range[np.argmin(davies_bouldin_scores)]
axes[1, 0].axvline(best_k_db, color='green', linestyle='--', linewidth=2,
                   label=f'Best k={best_k_db}')
axes[1, 0].legend()

# Calinski-Harabasz score
axes[1, 1].plot(k_range, calinski_harabasz_scores, 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_title('Calinski-Harabasz Index (Higher is Better)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1, 1].set_ylabel('Calinski-Harabasz Score', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(k_range)
best_k_ch = k_range[np.argmax(calinski_harabasz_scores)]
axes[1, 1].axvline(best_k_ch, color='red', linestyle='--', linewidth=2,
                   label=f'Best k={best_k_ch}')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('plots/05_optimal_clusters_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/05_optimal_clusters_analysis.png")

print(f"\n🎯 RECOMMENDED NUMBER OF CLUSTERS:")
print(f"  Best by Silhouette Score: k={best_k_silhouette}")
print(f"  Best by Davies-Bouldin: k={best_k_db}")
print(f"  Best by Calinski-Harabasz: k={best_k_ch}")

# Use the most common recommendation or default to a reasonable number
from collections import Counter
recommendations = [best_k_silhouette, best_k_db, best_k_ch]
optimal_k = Counter(recommendations).most_common(1)[0][0]
print(f"\n  ✓ Using k={optimal_k} clusters for comparative analysis")

# ============================================================================
# 6. APPLY MULTIPLE CLUSTERING ALGORITHMS
# ============================================================================
print("\n" + "=" * 100)
print("6. APPLYING MULTIPLE CLUSTERING ALGORITHMS")
print("=" * 100)

# Dictionary to store results
clustering_results = {}
clustering_metrics = {}

# 6.1 K-MEANS CLUSTERING
print("\n🔵 1. K-Means Clustering...")
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_scaled)
clustering_results['K-Means'] = labels_kmeans

# Calculate metrics
clustering_metrics['K-Means'] = {
    'Silhouette': silhouette_score(X_scaled, labels_kmeans),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, labels_kmeans),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, labels_kmeans)
}

print(f"  ✓ K-Means completed")
print(f"    Silhouette Score: {clustering_metrics['K-Means']['Silhouette']:.4f}")
print(f"    Davies-Bouldin Score: {clustering_metrics['K-Means']['Davies-Bouldin']:.4f}")
print(f"    Calinski-Harabasz Score: {clustering_metrics['K-Means']['Calinski-Harabasz']:.2f}")
print(f"    Cluster sizes: {np.bincount(labels_kmeans)}")

# 6.2 HIERARCHICAL CLUSTERING (AGGLOMERATIVE)
print("\n🟢 2. Hierarchical Clustering (Agglomerative)...")
hierarchical = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
labels_hierarchical = hierarchical.fit_predict(X_scaled)
clustering_results['Hierarchical'] = labels_hierarchical

clustering_metrics['Hierarchical'] = {
    'Silhouette': silhouette_score(X_scaled, labels_hierarchical),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, labels_hierarchical),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, labels_hierarchical)
}

print(f"  ✓ Hierarchical Clustering completed")
print(f"    Silhouette Score: {clustering_metrics['Hierarchical']['Silhouette']:.4f}")
print(f"    Davies-Bouldin Score: {clustering_metrics['Hierarchical']['Davies-Bouldin']:.4f}")
print(f"    Calinski-Harabasz Score: {clustering_metrics['Hierarchical']['Calinski-Harabasz']:.2f}")
print(f"    Cluster sizes: {np.bincount(labels_hierarchical)}")

# 6.3 GAUSSIAN MIXTURE MODEL (GMM)
print("\n🟡 3. Gaussian Mixture Model...")
gmm = GaussianMixture(n_components=optimal_k, random_state=42, n_init=10)
labels_gmm = gmm.fit_predict(X_scaled)
clustering_results['GMM'] = labels_gmm

clustering_metrics['GMM'] = {
    'Silhouette': silhouette_score(X_scaled, labels_gmm),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, labels_gmm),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, labels_gmm)
}

print(f"  ✓ GMM completed")
print(f"    Silhouette Score: {clustering_metrics['GMM']['Silhouette']:.4f}")
print(f"    Davies-Bouldin Score: {clustering_metrics['GMM']['Davies-Bouldin']:.4f}")
print(f"    Calinski-Harabasz Score: {clustering_metrics['GMM']['Calinski-Harabasz']:.2f}")
print(f"    Cluster sizes: {np.bincount(labels_gmm)}")

# 6.4 DBSCAN
print("\n🔴 4. DBSCAN (Density-Based)...")
# Tune eps using k-distance graph
from sklearn.neighbors import NearestNeighbors
neighbors = NearestNeighbors(n_neighbors=5)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)
distances = np.sort(distances[:, -1], axis=0)

# Use elbow point estimate
eps_estimate = np.percentile(distances, 95)
print(f"  Estimated eps parameter: {eps_estimate:.4f}")

dbscan = DBSCAN(eps=eps_estimate, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)
clustering_results['DBSCAN'] = labels_dbscan

# Handle noise points (-1 labels)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = list(labels_dbscan).count(-1)

if n_clusters_dbscan > 1:
    # Only calculate metrics if we have meaningful clusters
    labels_dbscan_clean = labels_dbscan[labels_dbscan != -1]
    X_scaled_clean = X_scaled[labels_dbscan != -1]

    if len(labels_dbscan_clean) > 0:
        clustering_metrics['DBSCAN'] = {
            'Silhouette': silhouette_score(X_scaled_clean, labels_dbscan_clean),
            'Davies-Bouldin': davies_bouldin_score(X_scaled_clean, labels_dbscan_clean),
            'Calinski-Harabasz': calinski_harabasz_score(X_scaled_clean, labels_dbscan_clean)
        }
    else:
        clustering_metrics['DBSCAN'] = {'Silhouette': np.nan, 'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan}
else:
    clustering_metrics['DBSCAN'] = {'Silhouette': np.nan, 'Davies-Bouldin': np.nan, 'Calinski-Harabasz': np.nan}

print(f"  ✓ DBSCAN completed")
print(f"    Clusters found: {n_clusters_dbscan}")
print(f"    Noise points: {n_noise} ({100*n_noise/len(labels_dbscan):.2f}%)")
if not np.isnan(clustering_metrics['DBSCAN']['Silhouette']):
    print(f"    Silhouette Score: {clustering_metrics['DBSCAN']['Silhouette']:.4f}")
    print(f"    Davies-Bouldin Score: {clustering_metrics['DBSCAN']['Davies-Bouldin']:.4f}")
    print(f"    Calinski-Harabasz Score: {clustering_metrics['DBSCAN']['Calinski-Harabasz']:.2f}")

# 6.5 MEAN SHIFT
print("\n🟣 5. Mean Shift...")
print("  ⚠️  Note: Mean Shift can be slow on large datasets...")
# Use a bandwidth estimate
from sklearn.cluster import estimate_bandwidth
bandwidth = estimate_bandwidth(X_scaled, quantile=0.2, n_samples=min(500, len(X_scaled)))
print(f"  Estimated bandwidth: {bandwidth:.4f}")

meanshift = MeanShift(bandwidth=bandwidth, bin_seeding=True)
labels_meanshift = meanshift.fit_predict(X_scaled)
clustering_results['MeanShift'] = labels_meanshift

n_clusters_ms = len(set(labels_meanshift))
clustering_metrics['MeanShift'] = {
    'Silhouette': silhouette_score(X_scaled, labels_meanshift),
    'Davies-Bouldin': davies_bouldin_score(X_scaled, labels_meanshift),
    'Calinski-Harabasz': calinski_harabasz_score(X_scaled, labels_meanshift)
}

print(f"  ✓ Mean Shift completed")
print(f"    Clusters found: {n_clusters_ms}")
print(f"    Silhouette Score: {clustering_metrics['MeanShift']['Silhouette']:.4f}")
print(f"    Davies-Bouldin Score: {clustering_metrics['MeanShift']['Davies-Bouldin']:.4f}")
print(f"    Calinski-Harabasz Score: {clustering_metrics['MeanShift']['Calinski-Harabasz']:.2f}")
print(f"    Cluster sizes: {np.bincount(labels_meanshift)}")

# ============================================================================
# 7. COMPARATIVE ANALYSIS OF CLUSTERING METHODS
# ============================================================================
print("\n" + "=" * 100)
print("7. COMPARATIVE ANALYSIS - ALL METHODS")
print("=" * 100)

# Create comparison DataFrame
comparison_df = pd.DataFrame(clustering_metrics).T
comparison_df = comparison_df.round(4)
print("\n📊 Clustering Performance Comparison:")
print(comparison_df.to_string())

# Save comparison table
comparison_df.to_csv('results/clustering_comparison.csv')
print("\n✓ Comparison table saved to: results/clustering_comparison.csv")

# VISUALIZATION 6: Metrics Comparison
print("\n📊 Creating: Clustering Methods Comparison...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Clustering Methods Performance Comparison', fontsize=16, fontweight='bold')

metrics_names = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']
colors_methods = sns.color_palette("Set2", len(clustering_results))

for idx, metric in enumerate(metrics_names):
    ax = axes[idx]
    methods = []
    scores = []

    for method in comparison_df.index:
        if not np.isnan(comparison_df.loc[method, metric]):
            methods.append(method)
            scores.append(comparison_df.loc[method, metric])

    bars = ax.bar(range(len(methods)), scores, color=colors_methods[:len(methods)],
                   edgecolor='black', linewidth=1.5)
    ax.set_title(f'{metric} Score', fontsize=13, fontweight='bold')
    ax.set_xlabel('Method', fontsize=11)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=45, ha='right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/06_clustering_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/06_clustering_metrics_comparison.png")


# ============================================================================
# 8. EVALUATE AND SELECT THE BEST CLUSTERING METHOD
# ============================================================================
print("\n" + "=" * 100)
print("8. EVALUATING AND SELECTING THE BEST CLUSTERING METHOD")
print("=" * 100)

# Rank methods based on metrics (considering direction: higher is better for Silhouette/Calinski-Harabasz, lower for Davies-Bouldin)
ranked_metrics = comparison_df.copy()
ranked_metrics['Silhouette_Rank'] = ranked_metrics['Silhouette'].rank(ascending=False)
ranked_metrics['Davies-Bouldin_Rank'] = ranked_metrics['Davies-Bouldin'].rank(ascending=True)
ranked_metrics['Calinski-Harabasz_Rank'] = ranked_metrics['Calinski-Harabasz'].rank(ascending=False)

# Calculate an overall rank (sum of ranks)
ranked_metrics['Overall_Rank'] = ranked_metrics[['Silhouette_Rank', 'Davies-Bouldin_Rank', 'Calinski-Harabasz_Rank']].sum(axis=1)
ranked_metrics = ranked_metrics.sort_values('Overall_Rank')

print("\n🏆 Clustering Method Ranking:")
print(ranked_metrics[['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz', 'Overall_Rank']].to_string())

best_method = ranked_metrics.index[0]
print(f"\nBased on overall ranking, the recommended method is: {best_method}")

# Select the labels from the best method
best_labels = clustering_results[best_method]
df['Cluster_Label'] = best_labels

print(f"\n✓ Assigned cluster labels from '{best_method}' to the dataset.")

# ============================================================================
# 9. ANALYZE AND CHARACTERIZE CLUSTERS
# ============================================================================
print("\n" + "=" * 100)
print("9. ANALYZING AND CHARACTERIZING CLUSTERS")
print("=" * 100)

# Add cluster labels to the original (or imputed) data for analysis
df_clustered = df.copy()
df_clustered['Cluster_Label'] = best_labels

# Analyze cluster sizes
cluster_sizes = df_clustered['Cluster_Label'].value_counts().sort_index()
print("\n📊 Cluster Sizes:")
print(cluster_sizes.to_string())

# Analyze cluster characteristics based on original (non-scaled) features
# Use the original numeric features for interpretation
original_numeric_features = [col for col in df.select_dtypes(include=[np.number]).columns.tolist()
                             if col not in ['customer_unique_id', 'Cluster_Label']]

cluster_summary = df_clustered.groupby('Cluster_Label')[original_numeric_features].mean().T
print("\n📈 Cluster Summary (Mean of Original Features):")
print(cluster_summary.to_string(float_format='%.2f'))

# Analyze categorical features per cluster (e.g., most frequent state/city)
if categorical_features:
    print("\n📊 Categorical Feature Distribution per Cluster:")
    for col in categorical_features:
        if col != 'customer_unique_id':
            print(f"\n  {col}:")
            # Get value counts for each cluster
            cluster_categorical_counts = df_clustered.groupby('Cluster_Label')[col].value_counts().unstack(fill_value=0)
            # Calculate percentages
            cluster_categorical_percentages = cluster_categorical_counts.apply(lambda x: 100 * x / x.sum(), axis=1)
            # Display top few for each cluster or overall
            print(cluster_categorical_percentages.head().to_string(float_format='%.2f'))

# VISUALIZATION 7: Cluster Profiles (Mean of Features)
print("\n📊 Creating: Cluster Profiles Visualization...")
plt.figure(figsize=(14, 8))
sns.heatmap(cluster_summary, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=.5)
plt.title('Cluster Profiles (Mean Feature Values)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Cluster Label', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig('plots/07_cluster_profiles_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("  ✓ Saved: plots/07_cluster_profiles_heatmap.png")


# VISUALIZATION 8: PCA Reduced Visualization of Clusters
print("\n📊 Creating: PCA Reduced Cluster Visualization...")

# Reduce data to 2 components for visualization using PCA
pca_viz = PCA(n_components=2, random_state=42)
X_pca_viz = pca_viz.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_viz[:, 0], X_pca_viz[:, 1], c=best_labels, cmap='viridis', s=10, alpha=0.6)

# Add centroids if using K-Means or GMM
if best_method == 'K-Means' and hasattr(kmeans, 'cluster_centers_'):
     # Transform centroids to PCA space
     centroids_pca = pca_viz.transform(kmeans.cluster_centers_)
     plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], marker='X', s=200,
                 color='red', edgecolor='black', linewidth=2, label='Centroids')

plt.title(f'Cluster Visualization (PCA Reduced) - {best_method}', fontsize=16, fontweight='bold')
plt.xlabel('Principal Component 1', fontsize=12)
plt.ylabel('Principal Component 2', fontsize=12)
plt.colorbar(scatter, label='Cluster Label')
if best_method == 'K-Means':
    plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(f'plots/08_pca_cluster_viz_{best_method.lower().replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ Saved: plots/08_pca_cluster_viz_{best_method.lower().replace(' ', '_')}.png")


# ============================================================================
# 10. SAVE RESULTS
# ============================================================================
print("\n" + "=" * 100)
print("10. SAVING FINAL RESULTS")
print("=" * 100)

# Save the dataframe with cluster labels
df_clustered[['customer_unique_id', 'Cluster_Label']].to_csv('results/customer_clusters.csv', index=False)
print("\n✓ Customer IDs with assigned cluster labels saved to: results/customer_clusters.csv")

# Save the cluster summary
cluster_summary.to_csv('results/cluster_summary_stats.csv')
print("✓ Cluster summary statistics saved to: results/cluster_summary_stats.csv")


print("\n" + "=" * 100)
print("COMPREHENSIVE CLUSTERING ANALYSIS COMPLETED")
print("=" * 100)

Created 'plots' directory
Created 'results' directory
OLIST CUSTOMER SEGMENTATION - COMPREHENSIVE CLUSTERING ANALYSIS

1. LOADING CLEANED CUSTOMER DATASET

✓ Dataset loaded successfully!
  Shape: (93358, 33)
  Customers: 93,358
  Features: 33

📊 Dataset Overview:
  Memory usage: 64.87 MB

📋 Column List:
   1. customer_unique_id
   2. num_orders
   3. total_spent
   4. avg_order_value
   5. avg_item_price_per_order
   6. avg_items_per_order
   7. avg_review
   8. avg_installments
   9. city
  10. state
  11. latitude
  12. longitude
  13. recency_days
  14. num_product_categories
  15. most_bought_category
  16. least_bought_category
  17. least_used_payment
  18. aov_range
  19. avg_item_price_range
  20. most_used_payment_boleto
  21. most_used_payment_credit_card
  22. most_used_payment_debit_card
  23. most_used_payment_voucher
  24. cat_pca_1
  25. cat_pca_2
  26. cat_pca_3
  27. cat_pca_4
  28. cat_pca_5
  29. cat_pca_6
  30. cat_pca_7
  31. cat_pca_8
  32. cat_pca_9
  33. cat_pca